# Optuna 優化實驗分析
**實驗：** `optuna_Llama-3.2-1B-Instruct_gsm8k_20260318_031558`

此 notebook 展示各 trial 相較於 baseline 的指標變化（%），並以顏色深淺表示差距大小。

In [19]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

EXP_DIR = Path("optuna_Llama-3.2-1B-Instruct_gsm8k_20260318_031558")

with open(EXP_DIR / "optimization_results.json") as f:
    results = json.load(f)

with open(EXP_DIR / "experiment_config.json") as f:
    config = json.load(f)

baseline = config["baseline"]
print("Baseline:")
for k, v in baseline.items():
    print(f"  {k}: {v:.4f}")

Baseline:
  accuracy: 0.3480
  latency: 1026.4961
  vram: 2.4242
  emissions: 0.0543


In [20]:
def summarize_config(cfg):
    """將 config dict 轉為簡短描述字串"""
    mode = cfg.get("mode", "unknown")
    parts = []

    if "quant" in cfg:
        q = cfg["quant"]
        method = q.get("method", "?")
        bits = q.get("bits", "?")
        gs = q.get("group_size", "?")
        fmt = q.get("format", "")
        dq = " dq" if q.get("double_quant") else ""
        q_str = f"{method} {bits}bit g{gs}"
        if fmt:
            q_str += f" {fmt}"
        q_str += dq
        parts.append(q_str)

    if "asvd" in cfg:
        a = cfg["asvd"]
        ratio = a.get("ratio", "?")
        alpha = a.get("alpha", "?")
        scaling = a.get("scaling", "?")
        parts.append(f"asvd r{int(ratio*100)} α{int(alpha*10)} {scaling}")

    if "sparse" in cfg:
        s = cfg["sparse"]
        struct = s.get("structure", "?")
        ratio = s.get("ratio", "")
        s_str = f"sparse {struct}"
        if ratio:
            s_str += f" {int(ratio*100)}%"
        parts.append(s_str)

    return " + ".join(parts) if parts else mode


rows = []
for trial in results:
    m = trial["metrics"]
    cfg = trial["config"]
    mode = cfg.get("mode", "unknown")

    acc_pct   = (m["accuracy"]  - baseline["accuracy"])  / baseline["accuracy"]  * 100
    lat_pct   = (m["latency"]   - baseline["latency"])   / baseline["latency"]   * 100
    vram_pct  = (m["vram"]      - baseline["vram"])      / baseline["vram"]      * 100
    emit_pct  = (m["emissions"] - baseline["emissions"]) / baseline["emissions"] * 100

    rows.append({
        "Trial": trial["iteration"],
        "名稱": trial["trial_name"],
        "Mode": mode,
        "設定": summarize_config(cfg),
        "Score": round(m["score"], 4),
        # 原始值
        "Accuracy": round(m["accuracy"], 4),
        "Latency (s)": round(m["latency"], 1),
        "VRAM (GB)": round(m["vram"], 3),
        "Emissions (g)": round(m["emissions"], 5),
        # 相較 baseline 的 %
        "Δ Accuracy %": round(acc_pct, 1),
        "Δ Latency %": round(lat_pct, 1),
        "Δ VRAM %": round(vram_pct, 1),
        "Δ Emissions %": round(emit_pct, 1),
    })

df = pd.DataFrame(rows).set_index("Trial")
print(f"共 {len(df)} 個 trials")

print("\nBaseline 參考值：")
print(f"  Accuracy : {baseline['accuracy']:.4f}")
print(f"  Latency  : {baseline['latency']:.1f} s")
print(f"  VRAM     : {baseline['vram']:.3f} GB")
print(f"  Emissions: {baseline['emissions']:.5f} g CO₂eq")

共 30 個 trials

Baseline 參考值：
  Accuracy : 0.3480
  Latency  : 1026.5 s
  VRAM     : 2.424 GB
  Emissions: 0.05429 g CO₂eq


In [21]:
def style_table(df):
    display_cols = ["名稱", "Mode", "設定", "Score",
                    "Accuracy", "Latency (s)", "VRAM (GB)", "Emissions (g)",
                    "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]
    d = df[display_cols]

    styled = d.style

    styled = styled.background_gradient(
        subset=["Score"], cmap="RdYlGn", vmin=d["Score"].min(), vmax=d["Score"].max()
    )
    styled = styled.background_gradient(
        subset=["Δ Accuracy %"], cmap="RdYlGn",
        vmin=d["Δ Accuracy %"].min(), vmax=d["Δ Accuracy %"].max()
    )
    for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
        styled = styled.background_gradient(
            subset=[col], cmap="RdYlGn_r",
            vmin=d[col].min(), vmax=d[col].max()
        )

    styled = styled.format({
        "Score": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Latency (s)": "{:.1f}",
        "VRAM (GB)": "{:.3f}",
        "Emissions (g)": "{:.5f}",
        "Δ Accuracy %": "{:+.1f}%",
        "Δ Latency %": "{:+.1f}%",
        "Δ VRAM %": "{:+.1f}%",
        "Δ Emissions %": "{:+.1f}%",
    })

    styled = styled.set_table_styles([
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                      ("font-size", "12px"), ("text-align", "center"),
                                      ("padding", "6px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"),
                                      ("text-align", "center")]},
        {"selector": "tr:hover td", "props": [("filter", "brightness(0.92)")]},
    ])

    return styled

styled_all = style_table(df)
styled_all.set_caption(
    f"全部 30 Trials — Baseline: Acc={baseline['accuracy']:.4f}, "
    f"Lat={baseline['latency']:.1f}s, VRAM={baseline['vram']:.3f}GB, "
    f"Emissions={baseline['emissions']:.5f}g"
)

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,0.2699,0.0136,3298.4,1.809,0.13367,-96.1%,+221.3%,-25.4%,+146.2%
2,trial_002_asvd_r090_a30_bnb_8bit,hybrid,bnb 8bit g128 + asvd r90 α3 fisher,0.8766,0.0144,473.8,3.905,0.01895,-95.9%,-53.8%,+61.1%,-65.1%
3,trial_003_gptq_8bit_g256_gptq_v2,quant_only,gptq 8bit g256 gptq_v2,0.2877,0.0205,2134.0,3.222,0.08411,-94.1%,+107.9%,+32.9%,+54.9%
4,trial_004_awq_4bit_g64,quant_only,awq 4bit g64,1.3163,0.2282,564.2,2.875,0.01654,-34.4%,-45.0%,+18.6%,-69.5%
5,trial_005_bnb_8bit,quant_only,bnb 8bit g128,1.2891,0.3548,566.8,4.197,0.02476,+2.0%,-44.8%,+73.1%,-54.4%
6,trial_006_bnb_4bit,quant_only,bnb 4bit g128,1.2898,0.3548,565.5,4.197,0.02475,+2.0%,-44.9%,+73.1%,-54.4%
7,trial_007_asvd_r085_a40,asvd_only,asvd r85 α4 fisher,2.6361,0.0152,137.9,3.933,0.00602,-95.6%,-86.6%,+62.2%,-88.9%
8,trial_008_sparse_70pct,sparse_only,sparse unstructured 70%,0.2965,0.0205,1619.4,6.989,0.06655,-94.1%,+57.8%,+188.3%,+22.6%
9,trial_009_asvd_r099_a50_bnb_8bit,hybrid,bnb 8bit g128 + asvd r99 α5 abs_mean,1.2116,0.3230,573.8,6.973,0.02461,-7.2%,-44.1%,+187.6%,-54.7%


In [22]:
from IPython.display import display
pd.set_option("display.max_colwidth", 60)
display(styled_all)

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,0.2699,0.0136,3298.4,1.809,0.13367,-96.1%,+221.3%,-25.4%,+146.2%
2,trial_002_asvd_r090_a30_bnb_8bit,hybrid,bnb 8bit g128 + asvd r90 α3 fisher,0.8766,0.0144,473.8,3.905,0.01895,-95.9%,-53.8%,+61.1%,-65.1%
3,trial_003_gptq_8bit_g256_gptq_v2,quant_only,gptq 8bit g256 gptq_v2,0.2877,0.0205,2134.0,3.222,0.08411,-94.1%,+107.9%,+32.9%,+54.9%
4,trial_004_awq_4bit_g64,quant_only,awq 4bit g64,1.3163,0.2282,564.2,2.875,0.01654,-34.4%,-45.0%,+18.6%,-69.5%
5,trial_005_bnb_8bit,quant_only,bnb 8bit g128,1.2891,0.3548,566.8,4.197,0.02476,+2.0%,-44.8%,+73.1%,-54.4%
6,trial_006_bnb_4bit,quant_only,bnb 4bit g128,1.2898,0.3548,565.5,4.197,0.02475,+2.0%,-44.9%,+73.1%,-54.4%
7,trial_007_asvd_r085_a40,asvd_only,asvd r85 α4 fisher,2.6361,0.0152,137.9,3.933,0.00602,-95.6%,-86.6%,+62.2%,-88.9%
8,trial_008_sparse_70pct,sparse_only,sparse unstructured 70%,0.2965,0.0205,1619.4,6.989,0.06655,-94.1%,+57.8%,+188.3%,+22.6%
9,trial_009_asvd_r099_a50_bnb_8bit,hybrid,bnb 8bit g128 + asvd r99 α5 abs_mean,1.2116,0.3230,573.8,6.973,0.02461,-7.2%,-44.1%,+187.6%,-54.7%


## Top 10 by Score

In [23]:
top10 = df.sort_values("Score", ascending=False).head(10)
display(style_table(top10).set_caption("Top 10 Trials（依 Score 排序）"))

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
10,trial_010_asvd_r085_a60_bnb_4bit,hybrid,bnb 4bit g128 dq + asvd r85 α6 fisher,4.4479,0.0144,72.7,6.721,0.00365,-95.9%,-92.9%,+177.3%,-93.3%
7,trial_007_asvd_r085_a40,asvd_only,asvd r85 α4 fisher,2.6361,0.0152,137.9,3.933,0.00602,-95.6%,-86.6%,+62.2%,-88.9%
15,trial_015_asvd_r090_a40,asvd_only,asvd r90 α4 abs_max,1.7912,0.0121,203.6,13.962,0.00869,-96.5%,-80.2%,+476.0%,-84.0%
4,trial_004_awq_4bit_g64,quant_only,awq 4bit g64,1.3163,0.2282,564.2,2.875,0.01654,-34.4%,-45.0%,+18.6%,-69.5%
6,trial_006_bnb_4bit,quant_only,bnb 4bit g128,1.2898,0.3548,565.5,4.197,0.02475,+2.0%,-44.9%,+73.1%,-54.4%
5,trial_005_bnb_8bit,quant_only,bnb 8bit g128,1.2891,0.3548,566.8,4.197,0.02476,+2.0%,-44.8%,+73.1%,-54.4%
25,trial_025_bnb_4bit,quant_only,bnb 4bit g128 dq,1.2526,0.3548,561.1,14.036,0.02464,+2.0%,-45.3%,+479.0%,-54.6%
18,trial_018_awq_4bit_g32,quant_only,awq 4bit g32,1.2119,0.2441,589.5,12.858,0.01816,-29.8%,-42.6%,+430.4%,-66.6%
9,trial_009_asvd_r099_a50_bnb_8bit,hybrid,bnb 8bit g128 + asvd r99 α5 abs_mean,1.2116,0.3230,573.8,6.973,0.02461,-7.2%,-44.1%,+187.6%,-54.7%


## 只看 Delta 欄（排序：Δ Accuracy %）

In [24]:
delta_df = df[["名稱", "Mode", "設定", "Score",
               "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]].sort_values(
    "Δ Accuracy %", ascending=False
)

styled_delta = delta_df.style
styled_delta = styled_delta.background_gradient(subset=["Score"], cmap="RdYlGn")
styled_delta = styled_delta.background_gradient(subset=["Δ Accuracy %"], cmap="RdYlGn")
for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
    styled_delta = styled_delta.background_gradient(subset=[col], cmap="RdYlGn_r")
styled_delta = styled_delta.format({
    "Score": "{:.4f}",
    "Δ Accuracy %": "{:+.1f}%",
    "Δ Latency %": "{:+.1f}%",
    "Δ VRAM %": "{:+.1f}%",
    "Δ Emissions %": "{:+.1f}%",
})
styled_delta = styled_delta.set_table_styles([
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                  ("font-size", "12px"), ("text-align", "center"),
                                  ("padding", "6px 10px")]},
    {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"),
                                  ("text-align", "center")]},
])
styled_delta.set_caption("依 Δ Accuracy % 排序（正值=優於 baseline）")

,名稱,Mode,設定,Score,Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,
5,trial_005_bnb_8bit,quant_only,bnb 8bit g128,1.2891,+2.0%,-44.8%,+73.1%,-54.4%
6,trial_006_bnb_4bit,quant_only,bnb 4bit g128,1.2898,+2.0%,-44.9%,+73.1%,-54.4%
25,trial_025_bnb_4bit,quant_only,bnb 4bit g128 dq,1.2526,+2.0%,-45.3%,+479.0%,-54.6%
9,trial_009_asvd_r099_a50_bnb_8bit,hybrid,bnb 8bit g128 + asvd r99 α5 abs_mean,1.2116,-7.2%,-44.1%,+187.6%,-54.7%
18,trial_018_awq_4bit_g32,quant_only,awq 4bit g32,1.2119,-29.8%,-42.6%,+430.4%,-66.6%
4,trial_004_awq_4bit_g64,quant_only,awq 4bit g64,1.3163,-34.4%,-45.0%,+18.6%,-69.5%
17,trial_017_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.7811,-56.6%,+1.1%,+485.8%,-50.6%
22,trial_022_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.7169,-57.3%,+17.2%,+484.5%,-44.2%
30,trial_030_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.4864,-58.4%,+72.8%,+484.5%,+23.7%


In [25]:
display(styled_delta)

,名稱,Mode,設定,Score,Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,
5,trial_005_bnb_8bit,quant_only,bnb 8bit g128,1.2891,+2.0%,-44.8%,+73.1%,-54.4%
6,trial_006_bnb_4bit,quant_only,bnb 4bit g128,1.2898,+2.0%,-44.9%,+73.1%,-54.4%
25,trial_025_bnb_4bit,quant_only,bnb 4bit g128 dq,1.2526,+2.0%,-45.3%,+479.0%,-54.6%
9,trial_009_asvd_r099_a50_bnb_8bit,hybrid,bnb 8bit g128 + asvd r99 α5 abs_mean,1.2116,-7.2%,-44.1%,+187.6%,-54.7%
18,trial_018_awq_4bit_g32,quant_only,awq 4bit g32,1.2119,-29.8%,-42.6%,+430.4%,-66.6%
4,trial_004_awq_4bit_g64,quant_only,awq 4bit g64,1.3163,-34.4%,-45.0%,+18.6%,-69.5%
17,trial_017_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.7811,-56.6%,+1.1%,+485.8%,-50.6%
22,trial_022_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.7169,-57.3%,+17.2%,+484.5%,-44.2%
30,trial_030_qqq_4bit_g-1,quant_only,qqq 4bit g-1,0.4864,-58.4%,+72.8%,+484.5%,+23.7%


## 統計摘要

In [26]:
summary = df.groupby("Mode")[["Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %", "Score"]].agg(["mean", "max", "min"]).round(1)
summary.style.background_gradient(cmap="coolwarm", axis=0).set_caption("各 Mode 統計摘要")

In [27]:
# 找出各指標最佳 trial
best_acc  = df.loc[df["Δ Accuracy %"].idxmax()]
best_lat  = df.loc[df["Δ Latency %"].idxmin()]
best_vram = df.loc[df["Δ VRAM %"].idxmin()]
best_emit = df.loc[df["Δ Emissions %"].idxmin()]
best_score = df.loc[df["Score"].idxmax()]

print("=== 各指標最佳 Trial ===")
print(f"最高 Accuracy 改善 : Trial {best_acc.name:>3} | {best_acc['名稱']} | Δ Acc={best_acc['Δ Accuracy %']:+.1f}%")
print(f"最低 Latency 增幅  : Trial {best_lat.name:>3} | {best_lat['名稱']} | Δ Lat={best_lat['Δ Latency %']:+.1f}%")
print(f"最低 VRAM 增幅     : Trial {best_vram.name:>3} | {best_vram['名稱']} | Δ VRAM={best_vram['Δ VRAM %']:+.1f}%")
print(f"最低 Emissions 增幅: Trial {best_emit.name:>3} | {best_emit['名稱']} | Δ Emit={best_emit['Δ Emissions %']:+.1f}%")
print(f"最高 Score         : Trial {best_score.name:>3} | {best_score['名稱']} | Score={best_score['Score']:.4f}")

=== 各指標最佳 Trial ===
最高 Accuracy 改善 : Trial   5 | trial_005_bnb_8bit | Δ Acc=+2.0%
最低 Latency 增幅  : Trial  10 | trial_010_asvd_r085_a60_bnb_4bit | Δ Lat=-92.9%
最低 VRAM 增幅     : Trial   1 | trial_001_gptq_8bit_g16_gptq | Δ VRAM=-25.4%
最低 Emissions 增幅: Trial  10 | trial_010_asvd_r085_a60_bnb_4bit | Δ Emit=-93.3%
最高 Score         : Trial  10 | trial_010_asvd_r085_a60_bnb_4bit | Score=4.4479
